# 07. Preprocessing Test

Verifies `preprocess_pose_dataframe()` against the synthetic squat baseline.

Pipeline position: Validation → Annotation → Exercise Definition → **Preprocessing** → Normalization

Completion criteria (docs/05 §Initial Completion Criteria):
1. `preprocessing.py` exists
2. coordinate columns selected from landmark list
3. visibility-based reliability marking
4. segment length and velocity checks produce reliability mask
5. exercise-aware swap detection — skip for bilateral_symmetric
6. short masked gaps can be interpolated
7. long masked gaps remain unresolved and reported
8. optional smoothing
9. frame and timestamp columns preserved
10. preprocessing report returned
11. pipeline.py runs preprocessing when enabled
12. this notebook verifies the behavior

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from movement.annotation import apply_annotation, load_annotation_csv
from movement.config import LANDMARKS, make_coordinate_columns, make_required_columns, make_visibility_columns
from movement.exercise_definition import load_exercise_definition
from movement.io import load_pose_csv
from movement.pipeline import (
    InterpolationConfig,
    PipelineConfig,
    PreprocessingConfig,
    ReliabilityConfig,
    SmoothingConfig,
    SwapDetectionConfig,
    run_pipeline,
)
from movement.preprocessing import preprocess_pose_dataframe
from movement.validation import run_basic_validation

print('imports OK')

## Data Setup

Mirrors the pipeline order: validation → annotation → exercise_definition → preprocessing.

In [ ]:
csv_path = '../data/pose/sample/mediapipe_squat_synthetic.csv'
ann_path = '../data/pose/sample/mediapipe_squat_synthetic_annotation.csv'
def_dir  = '../data/definitions/exercises'

df_raw = load_pose_csv(csv_path)
print(f'loaded: {df_raw.shape[0]} frames, {df_raw.shape[1]} columns')

In [ ]:
val_report = run_basic_validation(
    df=df_raw,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)
assert val_report['passed'], f'validation failed: {val_report}'
print(f'validation passed: {val_report["passed"]}')

In [ ]:
ann_df = load_annotation_csv(ann_path)
df_annotated, ann_report = apply_annotation(df_raw, ann_df)
print(f'annotated: exercise_type={df_annotated["exercise_type"].unique()}')
print(f'analysis frames: {ann_report["num_analysis_frames"]} / {ann_report["num_total_frames"]}')

In [ ]:
exercise_def = load_exercise_definition(
    exercise_id='squat',
    definitions_dir=def_dir,
)
print(f'exercise_id        : {exercise_def.exercise_id}')
print(f'laterality         : {exercise_def.classification["laterality"]}')
print(f'is_generic_fallback: {exercise_def.is_generic_fallback}')

## Direct Preprocessing Test

Call `preprocess_pose_dataframe()` directly (not via pipeline) for fine-grained inspection.

In [ ]:
pre_config = PreprocessingConfig(
    enabled=True,
    reliability=ReliabilityConfig(
        visibility_threshold=0.5,
        segment_length_tolerance=0.25,
        joint_angle_check=True,
        velocity_threshold_torso_per_sec=5.0,
    ),
    interpolation=InterpolationConfig(enabled=True, method='linear', max_gap_frames=3),
    smoothing=SmoothingConfig(enabled=False),
)

pre_df, pre_report = preprocess_pose_dataframe(
    df=df_annotated,
    landmarks=LANDMARKS,
    exercise_definition=exercise_def,
    config=pre_config,
)
print(f'output shape: {pre_df.shape}')
added = [c for c in pre_df.columns if c not in df_annotated.columns]
print(f'added {len(added)} columns: {added[:6]} ...')

## Check 1: Output Columns Present (criteria 1, 2, 9)

In [ ]:
for col in ('preprocessing_valid', 'preprocessing_note', 'swap_corrected'):
    assert col in pre_df.columns, f'missing: {col}'
print('PASS: frame-level output columns present')

missing_rel = [lm for lm in LANDMARKS if f'{lm}_x' in pre_df.columns
               and f'{lm}_reliable' not in pre_df.columns]
assert not missing_rel, f'missing _reliable: {missing_rel}'
n_lm = sum(1 for lm in LANDMARKS if f'{lm}_x' in pre_df.columns)
print(f'PASS: _reliable columns present for all {n_lm} landmarks')

for col in ('frame', 'timestamp'):
    assert col in pre_df.columns
print('PASS: frame and timestamp columns preserved (criteria 9)')

## Check 2: Baseline Squat — All Frames Reliable (criteria 3, 4)

In [ ]:
n_invalid = int((~pre_df['preprocessing_valid']).sum())
print(f'preprocessing_valid=False : {n_invalid} / {len(pre_df)}')
assert n_invalid == 0, f'Expected 0 invalid frames, got {n_invalid}'
print('PASS: all frames valid on clean squat baseline')

rel_cols = [f'{lm}_reliable' for lm in LANDMARKS if f'{lm}_x' in pre_df.columns]
unreliable = sum(int((~pre_df[c]).sum()) for c in rel_cols)
print(f'unreliable landmark-frames: {unreliable}')
assert unreliable == 0
print('PASS: all landmark-frames reliable on clean baseline')

## Check 3: Swap Detection Skipped for bilateral_symmetric (criteria 5)

In [ ]:
n_swap = int(pre_df['swap_corrected'].sum())
print(f'swap_corrected=True frames: {n_swap}')
assert n_swap == 0
print('PASS: no swap corrections (bilateral_symmetric → skip)')

assert pre_report['swap_detection_summary']['enabled'] is False
print('PASS: report.swap_detection_summary.enabled = False')

## Check 4: Interpolation — No Gaps in Clean Baseline (criteria 6, 7)

In [ ]:
itp = pre_report['interpolation_summary']
print(f'interpolation enabled   : {itp["enabled"]}')
print(f'short gaps interpolated : {itp["num_short_gaps_interpolated"]}')
print(f'long gaps unresolved    : {itp["num_long_gaps_unresolved"]}')
assert itp['num_short_gaps_interpolated'] == 0
assert itp['num_long_gaps_unresolved']    == 0
print('PASS: no gaps in clean baseline')

## Check 5: Smoothing Off by Default (criteria 8)

In [ ]:
sm = pre_report['smoothing_summary']
assert sm['enabled'] is False
assert sm['applied_columns'] == []
print(f'PASS: smoothing disabled by default, applied_columns=[]')

## Check 6: Full Report Structure (criteria 10)

In [ ]:
required_keys = [
    'method', 'exercise_type', 'pattern', 'laterality',
    'num_frames', 'num_coordinate_columns',
    'reliability_summary', 'swap_detection_summary',
    'interpolation_summary', 'smoothing_summary',
    'num_invalid_frames', 'applied_columns',
]
for k in required_keys:
    assert k in pre_report, f'missing: {k}'
print('PASS: all required report keys present')
print()
print(json.dumps(pre_report, indent=2, default=str)[:2000])

## Check 7: Report Values for Baseline Squat

In [ ]:
assert pre_report['exercise_type'] == 'squat'
assert pre_report['laterality']    == 'bilateral_symmetric'
assert pre_report['num_frames']    == len(pre_df)
assert pre_report['num_invalid_frames'] == 0

rel = pre_report['reliability_summary']
assert rel['num_segment_length_violations'] == 0
assert rel['num_joint_angle_violations']    == 0
assert rel['num_velocity_outliers']         == 0
assert rel['num_unreliable_landmark_frames'] == 0

print('PASS: all report values correct for clean squat baseline')

## Check 8: Pipeline Integration (criteria 11)

In [ ]:
import warnings

pipe_config = PipelineConfig()
pipe_config.preprocessing = PreprocessingConfig(enabled=True)

with warnings.catch_warnings(record=True):
    warnings.simplefilter('always')
    pipe_df, pipe_report = run_pipeline(
        df_annotated,
        config=pipe_config,
        landmarks=LANDMARKS,
    )

assert 'preprocessing' in pipe_report
print(f'PASS: preprocessing step in pipeline report')
print(f'steps executed: {list(pipe_report.keys())}')
print(f'pipeline output shape: {pipe_df.shape}')
print(f'pipeline preprocessing invalid frames: {pipe_report["preprocessing"]["num_invalid_frames"]}')

## Interpretation

Expected results for `mediapipe_squat_synthetic.csv` (clean baseline):

| Check | Expected |
|---|---|
| `preprocessing_valid` all True | 0 invalid frames |
| all `<lm>_reliable` True | 0 unreliable landmark-frames |
| `swap_corrected` all False | bilateral_symmetric → skip |
| `num_short_gaps_interpolated` | 0 (no gaps in clean data) |
| `num_long_gaps_unresolved` | 0 |
| `smoothing_summary.enabled` | False (default off) |

Abnormal variants (`squat_low_visibility`, `squat_segment_jump`, etc.) are needed
to trigger each detection branch. See docs/05a §6 for the synthetic variant list.